# Qwen3-TTS на Colab: план Б — torch/CUDA (`qwen-tts`)

Тот же самый замер RTF, но без MLX: официальный torch-путь, который рекомендует сам репозиторий
весов `siriusfreak/qwen3-tts-12hz-1.7b-ru-stress-cf` (в его README — ровно этот вызов,
`language="Russian"`, модели обучены в layout'е in-context cloning).

Зачем: MLX на CUDA — свежая ветка (0.32.x), и если на T4 он выдаст шум, NaN или плохой RTF, этот
ноутбук даёт рабочий путь и точку сравнения на той же карте.

Учтите: 1,7B fp16 = 4,6 ГБ веса, в float32 — ~9 ГБ из 15 доступных на T4.

**Перед запуском:** Runtime → Change runtime type → **T4 GPU**.

> Используйте только те образцы голоса, на которые у вас есть права.


## Setup

In [ ]:
# Тот же способ установки, что и в референсном блокноте esherma/CharacterVoiceCloning
!pip install -q git+https://github.com/QwenLM/Qwen3-TTS.git soundfile
# Альтернатива из PyPI: !pip install -q qwen-tts soundfile

In [ ]:
import torch
import soundfile as sf
from IPython.display import Audio, display

if not torch.cuda.is_available():
    print("⚠️  GPU не найден. Runtime → Change runtime type → T4 GPU.")
    device, dtype = "cpu", torch.float32
else:
    device = "cuda:0"
    # T4 (compute capability 7.5) не умеет bfloat16 — берём float32:
    # у float16 узкий диапазон и в acoustic code predictor бывают NaN.
    dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float32
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"  всего памяти: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} ГБ, dtype: {dtype}")

## Шаг 1. Образец голоса

WAV 5–15 с, один голос, без музыки. Берите `~/Downloads/colab-ref/narrator.wav`
(9,4 с, PCM float32, 24 кГц, моно) — он же используется в MLX-ноутбуке, так что цифры будут сравнимы.


In [ ]:
from google.colab import files

print("Выберите narrator.wav…")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())[0]
print("Образец:", ref_audio_path)
display(Audio(ref_audio_path))

## Шаг 2. Точная расшифровка образца и текст для синтеза

`ref_text` — слово в слово то, что сказано в образце (взят из `narrator.json`).
`text_to_generate` — тестовая фраза с ударениями. Здесь ударение — настоящий символ U+0301
**после** гласной; записан escape-последовательностями `\u0301`, чтобы комбинирующие символы
не потерялись при копировании. «+» перед гласной — фича скрипта репозитория, а не модели.


In [ ]:
ref_text = 'Усмехнулся тоже и черномазый. Белокурый несколько удивился, что ему удалось сказать довольно, впрочем, плохой каламбур.' # @param {type:"string"}

# Ударения — U+0301 после гласной:
text_to_generate = (
    "\u0417\u0430\u0301\u043c\u043e\u043a \u0441\u0442\u043e\u0438\u0301\u0442 \u043d\u0430 "
    "\u0445\u043e\u043b\u043c\u0435\u0301, \u0430 \u043c\u0430\u0301\u0441\u0442\u0435\u0440 "
    "\u043f\u043e\u0447\u0438\u043d\u0438\u0301\u043b \u0437\u0430\u043c\u043e\u0301\u043a "
    "\u0432 \u0434\u0432\u0435\u0440\u0438\u0301."
)
print(text_to_generate)
print("символов:", len(text_to_generate), "| U+0301 в тексте:", text_to_generate.count("\u0301"))

## Шаг 3. Загрузка модели

~4,6 ГБ fp16, первый раз ~2–4 мин. Модель — LoRA-дообучение под русские ударения, слитое в базовые
веса; в продакшене её гоняют как обычную Base-модель (у Base нет пресет-голосов — голос задаётся
только образцом + его текстом).


In [ ]:
import time

from qwen_tts import Qwen3TTSModel

t0 = time.time()
model = Qwen3TTSModel.from_pretrained(
    "siriusfreak/qwen3-tts-12hz-1.7b-ru-stress-cf",
    device_map=device,
    dtype=dtype,
)
print(f"✓ модель загружена за {time.time() - t0:.0f} с")

## Шаг 4. Синтез и замер RTF

In [ ]:
import numpy as np

t0 = time.time()
wavs, sr = model.generate_voice_clone(
    text=text_to_generate,
    language="Russian",
    ref_audio=ref_audio_path,
    ref_text=ref_text,
)
synth_s = time.time() - t0
audio = np.asarray(wavs[0], dtype=np.float32)
audio_s = len(audio) / sr

print(f"аудио {audio_s:.1f} с за {synth_s:.1f} с → RTF {synth_s / audio_s:.2f} "
      f"({audio_s / synth_s:.1f}× быстрее реального времени)")
print(f"проекция: книга 10 ч звука ≈ {10 * synth_s / audio_s:.1f} ч синтеза")

sf.write("out_torch.wav", audio, sr)
print("сохранено: out_torch.wav")
display(Audio("out_torch.wav"))

In [ ]:
from google.colab import files

files.download("out_torch.wav")

## Tips

- Повторный синтез без перезагрузки модели: обновите `text_to_generate` и запустите шаг 4 заново.
- Чтобы не кодировать образец на каждый вызов, сделайте `prompt = model.create_voice_clone_prompt(
  ref_audio=ref_audio_path, ref_text=ref_text)` и передавайте `voice_clone_prompt=prompt`
  в `generate_voice_clone` (в референсном блокноте это рекомендовано для длинных текстов).
- Многострочные тексты режьте по абзацам сами: у torch-пути нет нарезки, которая есть в
  `rab/qwen_tts_stress.py` (её `split_text` можно просто скопировать).
- Ударения: размечайте «+» перед гласной и переводите в U+0301 функцией `normalize_stress`
  из MLX-ноутбука — она одинаково годится для обоих бэкендов.
- Сравнивайте бэкенды на одном образце, одном тексте и одном seed: MLX и torch дают разные
  реализации сэмплинга, качество/скорость могут отличаться заметно.
